Weapons and Ammunition data analysis


In [11]:
import pandas as pd
pd.options.display.max_columns = None

df = pd.read_csv('data/amcdata_weapons_facilities_V2.csv', encoding='latin-1')
df = df[df['summary_category'] != 1]
weapons_df = df[df['item_type'] == 0]

### Select weapons life cycle stages

In [12]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',

    # End-of-life stages (no ban/restriction equivalents)
    'eliminitation',             # note: typo in the dataset
    'conversion',
    'modernization',
    'facility_destruction',
]

weapons_lifecycle_df = weapons_df[cols_to_keep]


Which columns are causing problems due to no variance?

In [13]:
ban_cols = [c for c in weapons_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in weapons_lifecycle_df.columns if 'restriction' in c]

flag_cols = ban_cols + restriction_cols

for col in flag_cols:
    unique_vals = weapons_lifecycle_df[col].dropna().unique()
    if len(unique_vals) <= 1:
        print(f"{col}: only contains {unique_vals}")

ban_possession: only contains [0]
ban_station: only contains [0]
ban_disposal: only contains [0]
restriction_development: only contains [0]
restriction_disposal: only contains [0]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [14]:
# Separate ban and restriction columns
ban_cols = [c for c in weapons_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in weapons_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = weapons_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                      NaN            -0.060248   
ban_testing                          NaN            -0.029361   
ban_production                       NaN            -0.081884   
ban_acquisition                      NaN            -0.223731   
ban_possession                       NaN                  NaN   
ban_station                          NaN                  NaN   
ban_transfer                         NaN            -0.081884   
ban_use                              NaN            -0.067958   
ban_disposal                         NaN                  NaN   

                 restriction_production  restriction_acquisition  \
ban_development               -0.178174                -0.070175   
ban_testing                   -0.086831                -0.034199   
ban_production                -0.242161                -0.095377   
ban_acquisition                0.329667                -0.260599   
ban_posse

In [15]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

                ban              restriction  correlation
19  ban_acquisition   restriction_production     0.329667
9       ban_testing   restriction_possession    -0.016667
6       ban_testing      testing_restriction    -0.029361
11      ban_testing          restriction_use    -0.029361
8       ban_testing  restriction_acquisition    -0.034199
3   ban_development   restriction_possession    -0.034199
33          ban_use   restriction_possession    -0.038576
27     ban_transfer   restriction_possession    -0.046481
15   ban_production   restriction_possession    -0.046481
0   ban_development      testing_restriction    -0.060248
5   ban_development          restriction_use    -0.060248
30          ban_use      testing_restriction    -0.067958
35          ban_use          restriction_use    -0.067958
2   ban_development  restriction_acquisition    -0.070175
32          ban_use  restriction_acquisition    -0.079156
10      ban_testing     restriction_transfer    -0.080246
24     ban_tra

In [16]:
# Pearson - default, fine for binary
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,0.487340,0.735767,0.269285,NaN,NaN,0.320195,0.886547,NaN,NaN,-0.060248,-0.178174,-0.070175,-0.034199,-0.164661,-0.060248,NaN
ban_testing,0.487340,1.000000,0.358569,0.131233,NaN,NaN,-0.046481,0.432049,NaN,NaN,-0.029361,-0.086831,-0.034199,-0.016667,-0.080246,-0.029361,NaN
ban_production,0.735767,0.358569,1.000000,0.263110,NaN,NaN,0.515873,0.642423,NaN,NaN,-0.081884,-0.242161,-0.095377,-0.046481,-0.223795,-0.081884,NaN
ban_acquisition,0.269285,0.131233,0.263110,1.000000,NaN,NaN,0.263110,0.184208,NaN,NaN,-0.223731,0.329667,-0.260599,-0.127000,-0.099514,-0.223731,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,0.320195,-0.046481,0.515873,0.263110,NaN,NaN,1.000000,0.267420,NaN,NaN,-0.081884,-0.131095,-0.095377,-0.046481,-0.223795,-0.081884,NaN
ban_use,0.886547,0.432049,0.642423,0.184208,NaN,NaN,0.267420,1.000000,NaN,NaN,-0.067958,-0.200976,-0.079156,-0.038576,-0.185733,-0.067958,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Spearman - better for ordinal/binary data, more robust
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,0.487340,0.735767,0.269285,NaN,NaN,0.320195,0.886547,NaN,NaN,-0.060248,-0.178174,-0.070175,-0.034199,-0.164661,-0.060248,NaN
ban_testing,0.487340,1.000000,0.358569,0.131233,NaN,NaN,-0.046481,0.432049,NaN,NaN,-0.029361,-0.086831,-0.034199,-0.016667,-0.080246,-0.029361,NaN
ban_production,0.735767,0.358569,1.000000,0.263110,NaN,NaN,0.515873,0.642423,NaN,NaN,-0.081884,-0.242161,-0.095377,-0.046481,-0.223795,-0.081884,NaN
ban_acquisition,0.269285,0.131233,0.263110,1.000000,NaN,NaN,0.263110,0.184208,NaN,NaN,-0.223731,0.329667,-0.260599,-0.127000,-0.099514,-0.223731,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,0.320195,-0.046481,0.515873,0.263110,NaN,NaN,1.000000,0.267420,NaN,NaN,-0.081884,-0.131095,-0.095377,-0.046481,-0.223795,-0.081884,NaN
ban_use,0.886547,0.432049,0.642423,0.184208,NaN,NaN,0.267420,1.000000,NaN,NaN,-0.067958,-0.200976,-0.079156,-0.038576,-0.185733,-0.067958,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
